In [1]:
import os
import json
import torch
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from datasets import load_dataset
import re
import numpy as np
from sklearn.linear_model import LogisticRegression

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [ ]:
# -----------------------------------------------------
#                CONFIG / SETUP
# -----------------------------------------------------
# --------------------------------------------------------
# CONFIG
# --------------------------------------------------------
DATA_DIR = "../data/pt/"
EMBED_KEY = "hidden_states_last_layer"

cache_dir="../models/"
output_dir = Path("../data/pt/")
output_dir.mkdir(parents=True, exist_ok=True)

os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HF_HUB_DISABLE_XET'] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = "..."
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [3]:
# check available GPUs
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

GPU 0: NVIDIA H100
GPU 1: NVIDIA H100


In [4]:
# load data
ds = load_dataset("TimSchopf/RINoBench")
print(ds)
labels = load_dataset("TimSchopf/RINoBench", "class_descriptions")
label_descriptions = [str(l['label'])+": "+l['description'] for l in labels['class_descriptions']]
print("Label descriptions:", label_descriptions)

DatasetDict({
    train: Dataset({
        features: ['source', 'venueid', 'research_idea', 'novelty_score', 'novelty_reasoning', 'related_works'],
        num_rows: 1104
    })
    test: Dataset({
        features: ['source', 'venueid', 'research_idea', 'novelty_score', 'novelty_reasoning', 'related_works'],
        num_rows: 277
    })
})
Label descriptions: ['1: The idea is not novel. All aspects already exist in prior work.', '2: The idea is marginally novel. It represents only a minor variation of existing work.', '3: The idea is somewhat novel. Aspects already exist in prior work. However, it might combine known approaches in new ways, apply them to new contexts, or propose incremental updates.', '4: The idea is novel. It introduces new aspects not present in existing work.', '5: The idea is highly innovative and novel. It is not present in existing work and potentially encourages new thinking or opens up new research directions.']


In [5]:
#model_id = "meta-llama/Llama-3.1-70B-Instruct"
#model_id = "Qwen/Qwen3-32B"
#model_id = "google/gemma-3-27b-it"
model_id = "openai/gpt-oss-20b"

# get compute capability
if torch.cuda.is_available():
    compute_capability = torch.cuda.get_device_capability()
    if compute_capability[0] >= 8:
        dtype = torch.bfloat16
    else:
        dtype = torch.float16
else:
    if torch.cpu.is_available() and hasattr(torch.cpu, 'is_bf16_supported') and torch.cpu.is_bf16_supported():
        dtype = torch.bfloat16
    else:
        dtype = torch.float32

# tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True, device_map='auto')
model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir, dtype=dtype, trust_remote_code=True, device_map='auto')

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

In [6]:
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    for device_id in range(num_devices):
        print(f"Device {device_id}: {torch.cuda.get_device_name(device_id)}")
        print(f"  Memory Allocated (GB): {torch.cuda.memory_allocated(device_id) / 1024**3:.2f}")
        print(f"  Memory Reserved (GB): {torch.cuda.memory_reserved(device_id) / 1024**3:.2f}")
        free, total = torch.cuda.mem_get_info(device_id)
        print(f"  Free Memory (GB): {free / 1024**3:.2f}")
        print(f"  Total Memory (GB): {total / 1024**3:.2f}")
else:
    print("CUDA is not available.")

Device 0: NVIDIA H100
  Memory Allocated (GB): 5.53
  Memory Reserved (GB): 7.32
  Free Memory (GB): 85.16
  Total Memory (GB): 93.09
Device 1: NVIDIA H100
  Memory Allocated (GB): 7.31
  Memory Reserved (GB): 8.92
  Free Memory (GB): 83.56
  Total Memory (GB): 93.09


In [7]:
def get_target_index(length, percent):
    """Helper to calculate zero-based index from percentage (0.0 to 1.0)."""
    if length <= 0:
        return 0
    percent = max(0.0, min(1.0, percent))
    return int(percent * (length - 1))

def get_reasoning_length(data, tokenizer, model_name):
    """
    Helper to calculate the number of tokens in the reasoning/thinking phase 
    based on the model type and data structure.
    """
    reasoning_tokens = 0
    
    # Logic extracted from your original snippet
    if model_name in ["openai-gpt-oss-20b", "openai-gpt-oss-120b"]:
        text_content = data['parsed_outputs'][0].content[0].text
        # Construct the specific prompt structure for counting
        full_text = "<|channel|>analysis<|message|>" + text_content + "<|end|>"
        reasoning_tokens = tokenizer(
            full_text,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]

    elif model_name in ["google-gemma-3-27b-it", "google-gemma-3-12b-it", "google-gemma-3-4b-it", "google-gemma-3-1b-it", "meta-llama-Llama-3.1-8B-Instruct", "meta-llama-Llama-3.1-70B-Instruct"]:
        # Split by review tag
        text_segment = data['parsed_outputs'].split("<REVIEW>")[0]
        reasoning_tokens = tokenizer(
            text_segment,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]

    elif model_name in ["Qwen-Qwen3-32B", "Qwen-Qwen3-14B", "Qwen-Qwen3-8B", "Qwen-Qwen3-4B"]:
        text_segment = data['parsed_outputs']['think_tokens']
        reasoning_tokens = tokenizer(
            text_segment,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]
        
    return reasoning_tokens

def load_response_token_embedding(path, tokenizer, model_name, position_percent=1.0):
    """
    Load embedding at X% of the *actual response* (excluding reasoning).
    0.0 = First token AFTER reasoning.
    1.0 = Last token of the file.
    """
    try:
        data = torch.load(path, map_location="cpu", weights_only=False)

        if EMBED_KEY not in data:
            print(f"Warning: {path} missing key '{EMBED_KEY}'. Skipped.")
            return None

        hs_list = data[EMBED_KEY]
        if not isinstance(hs_list, list) or len(hs_list) == 0:
            print(f"Warning: {path} has empty hidden-state list. Skipped.")
            return None

        # 1. Calculate the length of the reasoning prefix
        reasoning_len = get_reasoning_length(data, tokenizer, model_name)
        
        # Safety: Reasoning length cannot exceed total length
        if reasoning_len > len(hs_list):
            reasoning_len = len(hs_list)

        # 2. Define the 'Response' segment
        # The response starts at index `reasoning_len` and goes to the end
        response_len = len(hs_list) - reasoning_len

        if response_len <= 0:
            print(f"Warning: {path} has no response tokens (Reasoning len {reasoning_len} >= Total {len(hs_list)}). Skipped.")
            return None

        # 3. Calculate index relative to the response segment
        relative_idx = get_target_index(response_len, position_percent)
        
        # 4. Shift index by the reasoning length
        final_idx = reasoning_len + relative_idx
        
        target_h = hs_list[final_idx]
        target_h = target_h.squeeze(0).squeeze(0)

        del data
        del hs_list
        return target_h

    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

def load_reasoning_token_embedding(path, tokenizer, model_name, position_percent=1.0):
    """
    Load embedding at X% of the *reasoning sequence*.
    0.0 = First reasoning token.
    1.0 = Last reasoning token.
    """
    try:
        data = torch.load(path, map_location="cpu", weights_only=False)

        if EMBED_KEY not in data:
            print(f"Warning: {path} missing key '{EMBED_KEY}'. Skipped.")
            return None

        hs_list = data[EMBED_KEY]
        if not isinstance(hs_list, list) or len(hs_list) == 0:
            print(f"Warning: {path} has empty hidden-state list. Skipped.")
            return None

        # 1. Get reasoning length
        reasoning_len = get_reasoning_length(data, tokenizer, model_name)

        # Safety:
        if reasoning_len > len(hs_list):
            print(f"Warning: Reasoning token count {reasoning_len} exceeds hidden states {len(hs_list)}. Using max available.")
            reasoning_len = len(hs_list)
        
        if reasoning_len == 0:
            print(f"Warning: {path} has 0 reasoning tokens. Skipped.")
            return None

        # 2. Calculate index within the reasoning segment (0 to reasoning_len - 1)
        target_idx = get_target_index(reasoning_len, position_percent)

        # 3. if index is 0, ensure we get the first reasoning token by setting it + 1 (since index 0 returns embeddings of input token before reasoning/generation starts)
        if target_idx == 0:
            target_idx+=1
        
        target_reasoning_h = hs_list[target_idx]
        target_reasoning_h = target_reasoning_h.squeeze(0).squeeze(0)

        del data
        del hs_list
        return target_reasoning_h

    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

def load_all_embeddings(approach_name: str, model_name: str, tokenizer, use_reasoning_token=False, position_percent=1.0):
    print(f"Loading train + test embeddings from: {DATA_DIR} for model: {model_name} and approach: {approach_name} at position {position_percent*100}% (using reasoning token: {use_reasoning_token})")

    all_files = sorted([
        f for f in os.listdir(DATA_DIR)
        if f.endswith(".pt")
    ])

    # Regex patterns
    train_re = re.compile(approach_name+"_train_"+model_name+r"_\d+\.pt")
    test_re  = re.compile(approach_name+"_test_"+model_name+r"_\d+\.pt")

    x_train = []
    x_test = []

    for fname in tqdm(all_files, desc="Processing .pt files"):
        path = os.path.join(DATA_DIR, fname)
        emb = None

        if train_re.match(fname) or test_re.match(fname):
            if use_reasoning_token:
                # Load X% of reasoning
                emb = load_reasoning_token_embedding(path, tokenizer, model_name, position_percent=position_percent)
            else:
                # Load X% of response (excluding reasoning)
                emb = load_response_token_embedding(path, tokenizer, model_name, position_percent=position_percent)
            
            if emb is not None:
                if train_re.match(fname):
                    x_train.append(emb)
                else:
                    x_test.append(emb)

    return x_train, x_test

In [8]:
x_train, x_test = load_all_embeddings(approach_name="nle_only", model_name=model_id.replace("/","-"), tokenizer=tokenizer, use_reasoning_token=True, position_percent=1.0)

Loading train + test embeddings from: ../data/pt/ for model: openai-gpt-oss-20b and approach: nle_only at position 100.0% (using reasoning token: True)


Processing .pt files: 100%|██████████| 46468/46468 [05:22<00:00, 144.11it/s]  


In [9]:
think_tokens_list = []
for i in range(len(x_test)):
    o = torch.load("../data/pt/"+"nle_only"+"_test_"+model_id.replace("/","-")+"_"+str(i)+".pt", map_location="cpu", weights_only=False)['parsed_outputs']
    if model_id in ["Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "Qwen/Qwen3-14B", "Qwen/Qwen3-32B"]:
        think_tokens_list.append(o['think_tokens'].split("</think>")[0])
    elif model_id in ["meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-70B-Instruct", "google/gemma-3-4b-it", "google/gemma-3-12b-it", "google/gemma-3-27b-it"]:
        think_tokens_list.append(o.split("<REVIEW>")[0])
    elif model_id in ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]:
        for message in o:
            if message.channel == "analysis":
                think_tokens_list.append(message.content[0].text)

In [10]:
think_tokens_list[100]

'We need review: novelty level. Idea is max-zero-one labeling trick to encode subgraph membership into plain GNN. Related papers include SUB-GNN, SUGAR, G-Meta, ID-GNN, etc. Max-zero-one labeling is simple labeling. Novelty likely marginal to somewhat novel. Provide concise review ~80 words.'

In [11]:
y_train = list(ds["train"]["novelty_score"])
y_test = list(ds["test"]["novelty_score"])

print("converting data to numpy arrays...")
#convert to compatible values for sklearn
x_train = np.stack([t.to(torch.float32).cpu().numpy() for t in x_train])
x_test = np.stack([t.to(torch.float32).cpu().numpy() for t in x_test])
y_train = np.array(y_train)
y_test = np.array(y_test)

# train logistic regression probing classifier
print("Training probing classifier...")
clf = LogisticRegression(
    max_iter=100,
    solver="lbfgs"
).fit(x_train, y_train)

# predict on test set using probing classifier
y_pred = clf.predict(x_test)

converting data to numpy arrays...
Training probing classifier...


/home/tisc207h/workspaces/horse/conda/py3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
# get few-shot examples
example_1 = ds["train"].filter(lambda ex: ex["source"] == 'https://openreview.net/forum?id=Wac06sAkHk')[0]
relevant_related_works = ["A Continual Learning Survey: Defying Forgetting in Classification Tasks","Task-agnostic Continual Learning with Hybrid Probabilistic Models"]
relevant_related_works = [w.lower() for w in relevant_related_works]
rel_works = []
for w in example_1['related_works']:
    if w['title'].lower() in relevant_related_works:
        rel_works.append(w)
if len(set([w['title'] for w in rel_works])) == len(set(relevant_related_works)):
    print("Check:", True)
else:
    print("Check:", False)
    print("Not parsed related works:",set(relevant_related_works) - set([w['title'].lower() for w in rel_works]))

example_1['related_works'] = rel_works

example_2 = ds["train"].filter(lambda ex: ex["source"] == 'https://openreview.net/forum?id=zYWtq_HUCoi')[0]
relevant_related_works = ["Learning to Prune Deep Neural Networks via Layer-wise Optimal Brain Surgeon","Optimal Brain Compression: A Framework for Accurate Post-Training Quantization and Pruning"]
relevant_related_works = [w.lower() for w in relevant_related_works]
rel_works = []
for w in example_2['related_works']:
    if w['title'].lower() in relevant_related_works:
        rel_works.append(w)
if len(set([w['title'] for w in rel_works])) == len(set(relevant_related_works)):
    print("Check:", True)
else:
    print("Check:", False)
    print("Not parsed related works:",set(relevant_related_works) - set([w['title'].lower() for w in rel_works]))

example_2['related_works'] = rel_works

example_3 = ds["train"].filter(lambda ex: ex["source"] == 'https://openreview.net/forum?id=zlwBI2gQL3K')[0]
relevant_related_works = ["Contrastive Multi-View Representation Learning on Graphs","Learning Entity and Relation Embeddings for Knowledge Graph Completion"]
relevant_related_works = [w.lower() for w in relevant_related_works]
rel_works = []
for w in example_3['related_works']:
    if w['title'].lower() in relevant_related_works:
        rel_works.append(w)
if len(set([w['title'] for w in rel_works])) == len(set(relevant_related_works)):
    print("Check:", True)
else:
    print("Check:", False)
    print("Not parsed related works:",set(relevant_related_works) - set([w['title'].lower() for w in rel_works]))

example_3['related_works'] = rel_works

example_4 = ds["train"].filter(lambda ex: ex["source"] == 'https://openreview.net/forum?id=vuD2xEtxZcj')[0]
example_4['related_works'] = example_4['related_works'][:2]

example_5 = ds["train"].filter(lambda ex: ex["source"] == 'https://openreview.net/forum?id=zEn1BhaNYsC')[0]
example_5['related_works'] = example_4['related_works'][:2]

few_shot_examples = [example_1, example_2, example_3, example_4, example_5]

Check: True
Check: True
Check: True


In [13]:
def built_novelty_nle_prompt(idea, similar_documents, few_shot_examples, class_descriptions: list):

    class_descriptions = [descr.split(": ")[1] for descr in class_descriptions]
    class_desc_str = "\n       - " + "\n       - ".join(class_descriptions)
    example_str = ""
    for example in few_shot_examples:
        example_str += "<IDEA>" + str(example['research_idea']) + "</IDEA>" + "\n"
        for i,paper in enumerate(example['related_works']):
            example_str += f"<PAPER> Paper ID [{i}]: Title: {paper['title']}. Abstract: {paper['abstract']} </PAPER>" + "\n"

        example_str += f"<REVIEW> {example['novelty_reasoning']} </REVIEW>"
        example_str += "\n\n"
    
    relevant_papers = [{
                "role": "user",
                "content": "",
            }]
    for i,d in enumerate(similar_documents):
        relevant_papers[0]['content'] += f"<PAPER> Paper ID [{i}]: Title: {d['title']}. Abstract: {d['abstract']} </PAPER>/n"
        
    relevant_papers[0]['content'] += "First, before you do anything else, think and reason step-by-step via chain-of-thought! Then generate the review (<REVIEW> concise review </REVIEW>)"

    prompt = [
        {
            "role": "system",
            "content": "You are ReviewerGPT, an intelligent assistant that helps researchers evaluate the novelty of their ideas.",
        },
        {
            "role": "user",
            "content": f"""You are given some papers similar to the proposed idea (<IDEA> and </IDEA>). Your task is to evaluate the idea's novelty using the related papers (<PAPER> and </PAPER>) only.

                Types of novelty categories:
                - Not Novel: The idea closely replicates existing work with minimal or no new contributions.
                - Novel:
                    - The idea introduces new concepts or approaches that are not common in existing literature.
                    - The idea uniquely combines concepts from existing papers, but this combination does not occur in any related papers.
                    - A new application with same approach is also novel.

                Instructions:
                - Use the example reviews below to write a review for the provided idea by comparing it to the related papers.
                - Don't assume any prior knowledge about the idea.
                - When referencing a related paper, then use paper id in the review, mention it in this format: [5]. The paper ID is present between Paper ID [<paper_id>]: Title.
                - For reviewing, consider the following novelty categories: {class_desc_str}
                - Make sure the generated review follows the format in example reviews provided below.
                - The review should be concise - around 60 to 100 words.
                - Think step-by-step before generating the final review!



                {example_str}

                Output Format:
                <REVIEW> concise review </REVIEW>
                """,
        },
        {"role": "assistant", "content": "Sure, please provide the IDEA."},
        {"role": "user", "content": f"Here is the idea: <IDEA> {idea} </IDEA>"},
        {"role": "assistant", "content": "Okay, now provide the related papers."},
    ]

    prompt.extend(relevant_papers)

    return prompt

In [14]:
lower_first_char = lambda s: s[:1].lower() + s[1:] if s else ''

inputs = []
for i in range(len(x_test)):
    messages = built_novelty_nle_prompt(
            idea=ds['test']['research_idea'][i],
            similar_documents=ds['test']['related_works'][i][:25],
            few_shot_examples=few_shot_examples,
            class_descriptions=label_descriptions
        )
    
    if model_id in ["Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "Qwen/Qwen3-14B", "Qwen/Qwen3-32B"]:
        probing_prediction = f" Wait, upon further investigation {lower_first_char(label_descriptions[y_pred[i]-1].split(": ")[1])}\n</think>\n\n"
        messages.append({"role": "assistant", "content": think_tokens_list[i]+probing_prediction})
        
    elif model_id in ["meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-70B-Instruct", "google/gemma-3-4b-it", "google/gemma-3-12b-it", "google/gemma-3-27b-it"]:
        probing_prediction = f" Wait, upon further investigation {lower_first_char(label_descriptions[y_pred[i]-1].split(": ")[1])}\n\n<REVIEW>"
        messages.append({"role": "assistant", "content": think_tokens_list[i]+probing_prediction})
    
    elif model_id in ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]:
        probing_prediction = f" Wait, upon further investigation {lower_first_char(label_descriptions[y_pred[i]-1].split(": ")[1])}<|end|>"
        messages.append({"role": "assistant", "thinking": think_tokens_list[i]+probing_prediction, "content":""})
        
    inputs.append(messages)

In [15]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map='auto')
r = pipe(inputs, max_new_tokens=5000)
#for response in r:
#    print(response[0]['generated_text'][-1]['content'])
#    print("---------")

Device set to use cuda:0


In [17]:
for response in r:
    print(response[0]['generated_text'][-1]['content'])
    print("---------")

<REVIEW> The proposed VAD debiasing method is novel. Existing calibration studies (e.g., temperature scaling, Dirichlet calibration, Bayesian binning) focus on probability calibration without addressing the maximization bias that arises when metrics are computed on the model‑selected item set. No related work discusses a variance‑adjusting correction for this bias or its robustness to covariate shift. Thus, the idea introduces a distinct, previously unseen solution to a non‑trivial calibration problem, warranting a high novelty rating. </REVIEW>
---------
<REVIEW> The proposal is somewhat novel: it couples an autoregressive GNN with iterative structure refinement to jointly design antibody CDR sequences and their global 3D conformation without a pre‑specified target structure. Existing work (e.g., RAbD [0], ML‑based antibody design [1], and sequence‑conditional Fold2Seq [2]) either assumes a fixed backbone or conditions on a known structure. By removing that prerequisite and iterativel

In [16]:
# save outputs
approach_name = "TPR"
split_name = "test"
for i in range(len(r)):
    outfile = output_dir / f"{approach_name}_{split_name}_{model_id.replace('/', '-')}_{i}.pt"
    torch.save(
        {
            "parsed_outputs": r[i][0]['generated_text'][-1]['content']
        },
        outfile
    )